# Syndrome affinity analysis

**Author: OpenAI GPT-6.** This notebook analyzes JSON Lines emitted by `analyze_syndrome`: it validates the shot records, summarizes syndrome and decoder outcomes, plots logical-error rates against Hamming weight and affinity concentration, and fits a logistic regression.

The executable stores **unnormalized** affinities. For each fired detector, this notebook excludes the diagonal, divides each off-diagonal affinity by the sum of its row, and computes `effective_partners = 1 / sum(normalized_affinity**2)`. A value near 1 means one partner dominates; larger values mean a flatter set of plausible partners. If the row sum is zero (including syndromes with fewer than two fired detectors), its effective-partner value is set to 0 as an explicit missing-partner sentinel. The syndrome mean and maximum include those zeros. In schema 2, odd-physical-weight syndromes include the boundary detector as an affinity row and column; its row also participates in these summaries. The heatmaps use the mean and maximum effective-partner values on the vertical axis, respectively.

## Setup

Set `JSONL_PATH` to a file produced by the C++ executable, then run all cells. This notebook needs NumPy, Matplotlib, and statsmodels. For example: `python -m pip install numpy matplotlib statsmodels`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

JSONL_PATH = Path("syndromes.jsonl")
Y_BINS = 12
MIN_SHOTS_PER_HEATMAP_BIN = 5

In [ ]:
# Validate the JSONL schema while reducing each affinity matrix to a few
# per-shot statistics. Raw matrices are not retained in notebook memory.
def load_syndromes(path):
    metadata = None
    summary = None
    rows = []
    seen_indices = set()

    with Path(path).open("r", encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON on line {line_number}: {exc}") from exc

            kind = record.get("type")
            if kind == "metadata":
                if metadata is not None or rows or summary is not None:
                    raise ValueError(f"Unexpected metadata on line {line_number}")
                metadata = record
                if metadata.get("schema") not in (1, 2):
                    raise ValueError(f"Unsupported schema: {metadata.get('schema')}")
                if (metadata["schema"] == 2 and
                        metadata.get("boundary_detector_id") != metadata.get("detector_count")):
                    raise ValueError("Schema 2 requires boundary_detector_id == detector_count")
                if metadata.get("affinity_normalized") is not False:
                    raise ValueError("Expected unnormalized affinity matrices")
                continue

            if kind == "summary":
                if metadata is None or summary is not None:
                    raise ValueError(f"Unexpected summary on line {line_number}")
                summary = record
                continue

            if kind != "shot" or metadata is None or summary is not None:
                raise ValueError(f"Unexpected record on line {line_number}: {kind!r}")

            weight = record.get("hamming_weight")
            detector_ids = record.get("detector_ids")
            raw_affinity = record.get("affinity")
            shot_index = record.get("shot_index")
            logical_error = record.get("logical_error")
            if not isinstance(weight, int) or isinstance(weight, bool) or weight < 0:
                raise ValueError(f"Invalid Hamming weight on line {line_number}")
            boundary_id = (metadata["boundary_detector_id"]
                           if metadata["schema"] == 2 else None)
            matrix_size = weight + (weight % 2 if boundary_id is not None else 0)
            if not isinstance(detector_ids, list) or len(detector_ids) != matrix_size:
                raise ValueError(f"Invalid affinity detector list on line {line_number}")
            physical_ids = detector_ids[:weight]
            if (any(not isinstance(i, int) or isinstance(i, bool) or
                    i < 0 or i >= metadata["detector_count"] for i in physical_ids)
                    or physical_ids != sorted(set(physical_ids))):
                raise ValueError(f"Invalid physical detector IDs on line {line_number}")
            if boundary_id is not None:
                expected_ids = physical_ids + ([boundary_id] if weight % 2 else [])
                if detector_ids != expected_ids or record.get("boundary_added") is not bool(weight % 2):
                    raise ValueError(f"Invalid boundary detector on line {line_number}")
            if not isinstance(raw_affinity, list) or len(raw_affinity) != matrix_size * matrix_size:
                raise ValueError(f"Affinity shape disagrees with detector_ids on line {line_number}")
            if not isinstance(shot_index, int) or shot_index < 0 or shot_index in seen_indices:
                raise ValueError(f"Invalid or repeated shot index on line {line_number}")
            if not isinstance(logical_error, bool):
                raise ValueError(f"Invalid logical-error value on line {line_number}")
            seen_indices.add(shot_index)

            affinity = np.asarray(raw_affinity, dtype=float).reshape(matrix_size, matrix_size)
            if not np.all(np.isfinite(affinity)) or np.any((affinity < 0) | (affinity > 1)):
                raise ValueError(f"Affinity must be finite and in [0, 1] on line {line_number}")
            if matrix_size:
                np.fill_diagonal(affinity, 0.0)
                row_sums = affinity.sum(axis=1)
                positive = row_sums > 0
                effective = np.zeros(matrix_size, dtype=float)
                if positive.any():
                    normalized = affinity[positive] / row_sums[positive, None]
                    effective[positive] = 1.0 / np.square(normalized).sum(axis=1)
                mean_effective = float(effective.mean())
                max_effective = float(effective.max())
                zero_rows = int((~positive).sum())
            else:
                mean_effective = max_effective = 0.0
                zero_rows = 0

            rows.append((shot_index, weight, logical_error, mean_effective,
                         max_effective, zero_rows))

    if metadata is None or summary is None:
        raise ValueError("Expected both metadata and final summary; file may be incomplete")
    if len(rows) != metadata["requested_shots"] or len(rows) != summary["completed_shots"]:
        raise ValueError("Shot count does not match metadata or summary")
    if seen_indices != set(range(len(rows))):
        raise ValueError("Shot indices must cover 0 through completed_shots - 1")

    dtype = [("shot_index", "i8"), ("hamming_weight", "i4"),
             ("logical_error", "?"), ("mean_effective_partners", "f8"),
             ("max_effective_partners", "f8"), ("zero_rows", "i4")]
    data = np.array(rows, dtype=dtype)
    return metadata, summary, data

metadata, summary, data = load_syndromes(JSONL_PATH)

In [ ]:
# Overview: sample size, decoder error rate, syndrome sizes, affinity
# concentration, and the prevalence of detectors with no connected partner.
shots = len(data)
errors = int(data["logical_error"].sum())
weights = data["hamming_weight"]
mean_eff = data["mean_effective_partners"]
max_eff = data["max_effective_partners"]
zero_rows = data["zero_rows"]
print(f"File: {JSONL_PATH}")
print(f"Code: distance {metadata['distance']}, rounds {metadata['rounds']}, "
      f"SI1000 p={metadata['physical_error_rate']}, "
      f"opposite-basis detectors={metadata['include_opposite_basis_detectors']}")
print(f"Shots: {shots:,}  |  MPI ranks: {metadata['mpi_ranks']}  |  seed: {metadata['seed']}")
print(f"PyMatching logical errors: {errors:,} / {shots:,} = {100 * errors / shots:.4f}%")
print(f"Hamming weight: median={np.median(weights):.0f}, "
      f"mean={weights.mean():.2f}, range={weights.min()}–{weights.max()}")
print(f"Mean effective partners: median={np.median(mean_eff):.3f}, "
      f"10th/90th percentile={np.percentile(mean_eff, [10, 90])}")
print(f"Maximum effective partners: median={np.median(max_eff):.3f}, "
      f"10th/90th percentile={np.percentile(max_eff, [10, 90])}")
print(f"Syndromes with fewer than two physical detectors: {(weights < 2).sum():,}")
if metadata["schema"] == 2:
    print(f"Odd-weight syndromes augmented with boundary: {(weights % 2 == 1).sum():,}")
print(f"Syndromes with at least one zero-affinity row: {(zero_rows > 0).sum():,}")
print(f"Total zero-affinity rows: {zero_rows.sum():,}")
print("Zero-affinity rows are encoded as 0 effective partners in the plots and model.")
print("Logical-error rate by Hamming weight (first 20 populated weights):")
for weight in np.unique(weights)[:20]:
    group = weights == weight
    count = int(group.sum())
    failures = int(data["logical_error"][group].sum())
    print(f"  HW {weight:3d}: {failures:5d} / {count:7d} = {100 * failures / count:7.3f}%")
if len(np.unique(weights)) > 20:
    print(f"  ... {len(np.unique(weights)) - 20} additional populated weights")

In [ ]:
# Plot exact integer Hamming-weight columns and continuous effective-partner
# bins. Each pixel is failures / shots in that bin; sparse bins are masked.
def error_rate_heatmap(data, value_field, title, y_bins=Y_BINS,
                       min_shots=MIN_SHOTS_PER_HEATMAP_BIN):
    weights = data["hamming_weight"].astype(int)
    values = data[value_field]
    failures = data["logical_error"].astype(float)
    min_weight, max_weight = int(weights.min()), int(weights.max())
    x_edges = np.arange(min_weight - 0.5, max_weight + 1.5)
    if np.isclose(values.min(), values.max()):
        y_edges = np.linspace(max(0.0, values.min() - 0.5),
                              values.max() + 0.5, 2)
    else:
        y_edges = np.linspace(values.min(), values.max(), y_bins + 1)
    counts, _, _ = np.histogram2d(weights, values, bins=(x_edges, y_edges))
    errors, _, _ = np.histogram2d(weights, values, bins=(x_edges, y_edges),
                                  weights=failures)
    rate = np.divide(100 * errors, counts, out=np.full_like(errors, np.nan),
                     where=counts >= min_shots)
    rate = np.ma.masked_invalid(rate.T)
    from matplotlib.colors import LogNorm
    fig, axes = plt.subplots(
        2, 1, figsize=(max(8, min(22, (max_weight - min_weight + 1) * 0.45)), 9),
        sharex=True, sharey=True, constrained_layout=True)
    error_ax, count_ax = axes
    error_mesh = error_ax.pcolormesh(
        x_edges, y_edges, rate, cmap="viridis", vmin=0, vmax=100, shading="flat")
    fig.colorbar(error_mesh, ax=error_ax, label="PyMatching logical-error rate (%)")
    count_image = np.ma.masked_where(counts.T == 0, counts.T)
    count_mesh = count_ax.pcolormesh(
        x_edges, y_edges, count_image, cmap="Blues",
        norm=LogNorm(vmin=1, vmax=max(1, int(counts.max()))), shading="flat")
    fig.colorbar(count_mesh, ax=count_ax, label="Shots per bin (log scale)")
    error_ax.set_title(f"{title}: logical-error rate (at least {min_shots} shots per visible bin)")
    count_ax.set_title("Shot counts; white bins have no shots")
    for ax in axes:
        ax.set_ylabel(title)
    count_ax.set_xlabel("Syndrome Hamming weight")
    count_ax.set_xticks(np.arange(
        min_weight, max_weight + 1,
        max(1, (max_weight - min_weight + 1) // 20)))
    plt.show()
    print(f"Visible error-rate bins: {int((counts >= min_shots).sum())}/{counts.size}; "
          f"shots in visible bins: {int(counts[counts >= min_shots].sum()):,}/{len(data):,}")


In [ ]:
error_rate_heatmap(data, "mean_effective_partners",
                   "Mean effective partners across fired detectors")

In [ ]:
error_rate_heatmap(data, "max_effective_partners",
                   "Maximum effective partners across fired detectors")

## Multivariate logistic regression

The outcome is the PyMatching logical-error indicator. Predictors are Hamming weight and the syndrome mean and maximum effective-partner values; every shot is included. Zero-affinity rows contribute the 0 sentinel defined above. The model reports **McFadden's pseudo R²**, because ordinary least-squares R² is not defined for logistic regression. Coefficients and odds ratios are per one-unit increase in each unscaled predictor. This is an association model; class imbalance and correlation among predictors can make estimates unstable. The cell detects one-class and fit failures.

In [ ]:
import statsmodels.api as sm
from statsmodels.tools.sm_exceptions import PerfectSeparationError

def roc_auc_from_scores(labels, scores):
    # Mann–Whitney AUC, with midranks for tied scores.
    labels = np.asarray(labels, dtype=bool)
    scores = np.asarray(scores, dtype=float)
    positives = int(labels.sum())
    negatives = len(labels) - positives
    if positives == 0 or negatives == 0:
        return float("nan")
    order = np.argsort(scores, kind="stable")
    sorted_scores = scores[order]
    ranks = np.empty(len(scores), dtype=float)
    start = 0
    while start < len(scores):
        end = start + 1
        while end < len(scores) and sorted_scores[end] == sorted_scores[start]:
            end += 1
        ranks[order[start:end]] = ((start + 1) + end) / 2.0
        start = end
    return (ranks[labels].sum() - positives * (positives + 1) / 2) / (positives * negatives)

features = np.column_stack([
    data["hamming_weight"],
    data["mean_effective_partners"],
    data["max_effective_partners"],
])
names = ["hamming_weight", "mean_effective_partners", "max_effective_partners"]
y = data["logical_error"].astype(int)
X = sm.add_constant(features, has_constant="add")
if y.min() == y.max():
    print(f"Cannot fit logistic regression: all {len(y):,} outcomes are {y[0]}.")
elif np.linalg.matrix_rank(X) < X.shape[1]:
    print("Cannot fit all requested predictors: design matrix is rank-deficient.")
else:
    try:
        model = sm.Logit(y, X)
        fit = model.fit(disp=False, maxiter=200)
        print(f"Observations: {fit.nobs:,.0f}; errors: {int(y.sum()):,}; "
              f"non-errors: {int((1-y).sum()):,}")
        print(f"Converged: {fit.mle_retvals.get('converged', False)}")
        print(f"Log-likelihood: {fit.llf:.6g}; null log-likelihood: {fit.llnull:.6g}")
        print(f"McFadden pseudo R²: {fit.prsquared:.6g}")
        print(f"Likelihood-ratio statistic: {fit.llr:.6g}; p-value: {fit.llr_pvalue:.6g}")
        print(f"AIC: {fit.aic:.6g}; BIC: {fit.bic:.6g}")
        predictions = fit.predict(X)
        print(f"In-sample ROC-AUC: {roc_auc_from_scores(y, predictions):.6g}")
        print("ROC-AUC is descriptive here; an independent test set is needed for prediction claims.")
        print(fit.summary(xname=["intercept"] + names))
        intervals = fit.conf_int()
        print()
        print("Odds ratios per unit (95% confidence intervals):")
        for i, name in enumerate(["intercept"] + names):
            print(f"  {name:30s} {np.exp(fit.params[i]):.6g} "
                  f"[{np.exp(intervals[i, 0]):.6g}, {np.exp(intervals[i, 1]):.6g}]")
    except (np.linalg.LinAlgError, ValueError, PerfectSeparationError) as exc:
        print(f"Logistic regression could not be fitted reliably: {exc}")